## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
# type: ignore

# Notebook to generate pseudotime for the NCS lineage from 10x single cell RNA data of the vSVZ of wildtype mice

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KernelDensity

import scanorama
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri
rpy2.robjects.numpy2ri.activate()
from rpy2.robjects.packages import importr
r_sleepwalk = importr('sleepwalk')

os.chdir(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger')

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80, facecolor='white', color_map="cividis_r")

## Load and describe the data

In [ ]:
counts_wildtypes = sc.read_10x_h5(f"{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/AggWildtypes/outs/count/filtered_feature_bc_matrix.h5")
counts_wildtypes.var_names_make_unique(join='-')

In [ ]:
print(counts_wildtypes, f"var:\n{counts_wildtypes.var}", f"obs:\n{counts_wildtypes.obs}", sep="\n\n")

## Normalization and data preprocessing

In [ ]:
# assign the batches based on cell barcodes. Each batch (sample) was aggregated via CellRanger aggr, which appends "-1", "-2", and so on.
batches = [barcode_batch.split("-")[1] for barcode_batch in counts_wildtypes.obs_names]
counts_wildtypes.obs["batch"] = batches

In [ ]:
counts_wildtypes.var['mt'] = counts_wildtypes.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(counts_wildtypes, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(counts_wildtypes, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
counts_wildtypes.obs.n_genes_by_counts.plot(kind="kde", xlim=(0,10000))

In [ ]:
x_space = counts_wildtypes.obs.n_genes_by_counts.sort_values().to_numpy().reshape(-1, 1)
kernel = KernelDensity(kernel="gaussian", bandwidth=100).fit(x_space)

y_space = kernel.score_samples(x_space)

In [ ]:
x_space = x_space.flatten()
y_space = y_space[(x_space > 500) & (x_space < 1500)]
x_space = x_space[(x_space > 500) & (x_space < 1500)]

In [ ]:
cutoff = x_space[np.argmin(y_space)]

fig, axs = plt.subplots(1, 2, figsize=(10,5))
sc.pl.violin(counts_wildtypes, 'n_genes_by_counts',
             jitter=0.4, log=False, ax=axs[0], show=False)
sc.pl.violin(counts_wildtypes, 'n_genes_by_counts',
             jitter=0.4, log=True, ax=axs[1], show=False)
axs[0].axhline(y=cutoff, linewidth=2, color="r")
axs[1].axhline(y=cutoff, linewidth=2, color="r")

In [ ]:
# filter out cells that have low gene coverage and genes that are extremely rare
sc.pp.filter_cells(counts_wildtypes, min_genes=cutoff)
sc.pp.filter_genes(counts_wildtypes, min_cells=3)

In [ ]:
sc.pp.calculate_qc_metrics(counts_wildtypes, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(counts_wildtypes, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
threshold_mito = 10
threshold_genes_per_cell = 4500

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sc.pl.scatter(counts_wildtypes, x='total_counts', y='pct_counts_mt', ax=ax1, show=False)
ax1.axhline(y=threshold_mito, color='r', linestyle='--')
sc.pl.scatter(counts_wildtypes, x='total_counts', y='n_genes_by_counts', ax=ax2, show=False)
ax2.axhline(y=threshold_genes_per_cell, color='r', linestyle='--')

In [ ]:
# remove cells that have too many mitochondrial genes expressed or too many total counts
counts_wildtypes = counts_wildtypes[counts_wildtypes.obs.n_genes_by_counts < threshold_genes_per_cell, :]
counts_wildtypes = counts_wildtypes[counts_wildtypes.obs.pct_counts_mt < threshold_mito, :]

In [ ]:
# counts_wildtypes.write_h5ad('$M26_DATA_ROOT/3utr/wildtypes_analysis/cellranger/wildtypes_counts_raw.h5ad')

### Doublet identification and filtering with Scrublet

In [ ]:
adata_raw = counts_wildtypes.copy()
adata_raw

In [ ]:
sce.pp.scrublet(adata=adata_raw)

In [ ]:
sce.pl.scrublet_score_distribution(adata=adata_raw)

In [ ]:
adata_raw.obs

In [ ]:
counts_wildtypes.obs = counts_wildtypes.obs.join(adata_raw.obs[["doublet_score", "predicted_doublet"]])
counts_wildtypes

In [ ]:
counts_wildtypes.obs["predicted_doublet"] = np.where(counts_wildtypes.obs.predicted_doublet, "yes", "no")
counts_wildtypes.obs

In [ ]:
counts_wildtypes = counts_wildtypes[counts_wildtypes.obs.predicted_doublet=="no"]

In [ ]:
counts_wildtypes.raw = counts_wildtypes

In [ ]:
# normalize the counts per cell so that each cell has the same total
sc.pp.normalize_per_cell(counts_wildtypes)
# log the data, which is necessary for later analyses.
sc.pp.log1p(counts_wildtypes)
# store the log data in a layer so that we can retrieve later even after regressing out variables and scaling.
counts_wildtypes.layers["lognorm"] = counts_wildtypes.X.copy()
# identify highly variable genes. These genes will most of the times automatically be used by later tools.
sc.pp.highly_variable_genes(
    counts_wildtypes,
    flavor='seurat',
    min_mean=0.0125,
    max_mean=4,
    batch_key="batch",
)
counts_wildtypes


In [ ]:
sc.pl.highly_variable_genes(counts_wildtypes)

In [ ]:
# filter the data to keep only highly variable genes. (in .raw there will still be the full data)
counts_wildtypes = counts_wildtypes[:, counts_wildtypes.var.highly_variable]
counts_wildtypes

### Regress out the number of detected genes and number of UMIs (maybe don't regress out the mito percentage)

In [ ]:
sc.pp.regress_out(
    counts_wildtypes,
    keys=["total_counts", "total_counts_mt"],
    # keys=["n_genes", "total_counts", "total_counts_mt"],
    n_jobs=20,
)

In [ ]:
counts_wildtypes.layers["lognorm_regressed"] = counts_wildtypes.X.copy()

In [ ]:
g2m_genes = ['Gas2l3', 'Nek2', 'Hmgb2', 'Cdca2', 'Cdk1', 'Kif11', 'Kif2c', 'Ube2c', 'Smc4', 'Birc5', 'Pimreg',
             'Cks2', 'Tpx2', 'Cdca8', 'Kif23', 'Ncapd2', 'Ckap2', 'Ndc80', 'Tubb4b', 'Ckap5', 'Tacc3', 'Top2a',
             'Cdc25c', 'Rangap1', 'Aurkb', 'Anp32e', 'Ctcf', 'Bub1', 'Nusap1', 'Ccnb2', 'Psrc1', 'Mki67',
             'Cenpe', 'Ckap2l', 'G2e3', 'Cdca3', 'Ect2', 'Gtse1', 'Ttk', 'Cdc20', 'Dlgap5', 'Cenpf', 'Nuf2',
             'Hmmr', 'Hjurp', 'Anln', 'Cks1b', 'Aurka', 'Lbr', 'Kif20b', 'Tmpo', 'Cenpa', 'Cbx5']
s_genes = ['Cdca7', 'Dtl', 'Exo1', 'Rfc2', 'Ubr7', 'Gmnn', 'Mcm5', 'Ccne2', 'Mcm4', 'Tipin', 'Rrm2', 'Pola1',
           'Chaf1b', 'Msh2', 'Tyms', 'Rad51', 'Casp8ap2', 'Blm', 'Pcna', 'Brip1', 'Uhrf1', 'Cdc6', 'Cenpu',
           'Slbp', 'Prim1', 'Rrm1', 'Fen1', 'Mcm6', 'Hells', 'Clspn', 'E2f8', 'Usp1', 'Polr1b', 'Rad51ap1',
           'Mcm2', 'Pold3', 'Gins2', 'Nasp', 'Wdr76', 'Dscc1', 'Mrpl36', 'Cdc45', 'Ung', 'Rpa2', 'Mcm7']

In [ ]:
print(f"Number of G2M genes: {len(g2m_genes)}\nNumber of S genes: {len(s_genes)}")

In [ ]:
# Log-transformation of data and scaling should always be performed before scoring
sc.pp.scale(counts_wildtypes, zero_center=True)
sc.tl.score_genes_cell_cycle(counts_wildtypes, s_genes=s_genes, g2m_genes=g2m_genes)

### Regress out cell cycle difference

In [ ]:
counts_wildtypes_cc_regressed = counts_wildtypes.copy()
del counts_wildtypes_cc_regressed.layers["lognorm_regressed"]

In [ ]:
counts_wildtypes_cc_regressed.obs["cell_cycle_diff"] = counts_wildtypes_cc_regressed.obs["S_score"] - counts_wildtypes_cc_regressed.obs["G2M_score"]
sc.pp.regress_out(counts_wildtypes_cc_regressed, ['cell_cycle_diff'], n_jobs=20)

In [ ]:
counts_wildtypes_cc_regressed.write_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_cellcyclereg.h5ad')
counts_wildtypes.write_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled.h5ad')

In [ ]:
counts_wildtypes.X = counts_wildtypes.layers["lognorm_regressed"]
del counts_wildtypes.layers["lognorm_regressed"]
counts_wildtypes.X

In [ ]:
adata_scanorama = counts_wildtypes.copy()
adata_list = [adata_scanorama[adata_scanorama.obs['batch'] == i] for i in adata_scanorama.obs['batch'].unique()]

corrected = scanorama.correct_scanpy(adata_list, return_dimred=True)
print("Finished correction")
counts_wildtypes = corrected[0].concatenate(*corrected[1:])

counts_wildtypes.obs = adata_scanorama.obs  # get back the full annotation
counts_wildtypes.X = counts_wildtypes.X.toarray()  # scanorama returns sparse matrix. Densify it again.
counts_wildtypes.raw = adata_scanorama.raw.to_adata()  # get back the normalized log data.
counts_wildtypes.layers["lognorm"] = adata_scanorama.layers["lognorm"]

## Dimensionality reduction and embedding

In [ ]:
# scale to unit variance and zero mean
sc.pp.scale(counts_wildtypes, zero_center=True)
# compute PCA
sc.tl.pca(counts_wildtypes, svd_solver='arpack', n_comps=50)

In [ ]:
sc.pl.pca(counts_wildtypes, color='S100b')

In [ ]:
sc.pl.pca(counts_wildtypes, color='phase')

In [ ]:
sc.pl.pca(counts_wildtypes, color=['Ube2c', 'Ung'])

In [ ]:
sc.pl.pca_variance_ratio(counts_wildtypes, log=True, show=50)

In [ ]:
# compute UMAP
sc.pp.neighbors(counts_wildtypes, n_neighbors=100, n_pcs=40, use_rep="X_scanorama")  # use_rep="X_scanorama"
sc.tl.umap(counts_wildtypes, min_dist=0.5)

In [ ]:
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (150)}):
    sc.pl.umap(counts_wildtypes, color='batch')

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(22, 10))

sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="1"], color='phase', ax=axes[0, 0], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="2"], color='phase', ax=axes[0, 1], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="3"], color='phase', ax=axes[0, 2], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="4"], color='phase', ax=axes[1, 0], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="5"], color='phase', ax=axes[1, 1], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="6"], color='phase', ax=axes[1, 2], show=False)

axes[0, 0].set_title('WT mouse 1')
axes[0, 1].set_title('WT mouse 2')
axes[0, 2].set_title('WT mouse 3')
axes[1, 0].set_title('WT mouse 4')
axes[1, 1].set_title('WT mouse 5')
axes[1, 2].set_title('WT mouse 6')

plt.tight_layout()

In [ ]:
sc.pl.umap(counts_wildtypes, color="S_score")
sc.pl.umap(counts_wildtypes, color="G2M_score")
sc.pl.umap(counts_wildtypes, color=['S100b', 'Ptprc', 'Cldn11'])

In [ ]:
sc.pl.umap(counts_wildtypes, color=['Ube2c', 'Ung', 'Cenpa', 'Hmgb2'])

In [ ]:
sc.pl.umap(counts_wildtypes, color=['Egfr', 'Ascl1', 'Thbs4'])

In [ ]:
sc.pl.umap(counts_wildtypes, color='Ifit1')

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(22, 10))

sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="1"], color='Dcx', ax=axes[0, 0], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="2"], color='Dcx', ax=axes[0, 1], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="3"], color='Dcx', ax=axes[0, 2], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="4"], color='Dcx', ax=axes[1, 0], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="5"], color='Dcx', ax=axes[1, 1], show=False)
sc.pl.umap(counts_wildtypes[counts_wildtypes.obs.batch=="6"], color='Dcx', ax=axes[1, 2], show=False)

axes[0, 0].set_title('WT mouse 1')
axes[0, 1].set_title('WT mouse 2')
axes[0, 2].set_title('WT mouse 3')
axes[1, 0].set_title('WT mouse 4')
axes[1, 1].set_title('WT mouse 5')
axes[1, 2].set_title('WT mouse 6')

plt.tight_layout()

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white', color_map="cividis_r")
sc.pl.umap(counts_wildtypes, color=['Aqp4', 'Cxcl14', 'Cpe', 'S100a6'], use_raw=False)

In [ ]:
"""# inspect the embedding with sleepwalk
umap_coords = counts_wildtypes.obsm["X_umap"]
nr, nc = umap_coords.shape
r_umap_matrix = robjects.r.matrix(umap_coords, nrow=nr, ncol=nc)

pca_coords = counts_wildtypes.obsm["X_pca"]
nr, nc = pca_coords.shape
r_feature_matrix = robjects.r.matrix(pca_coords, nrow=nr, ncol=nc)
r_sleepwalk.sleepwalk(r_umap_matrix, r_feature_matrix)"""

In [ ]:
# r_sleepwalk.sleepwalk(r_umap_matrix, r_feature_matrix)

### Check alternative parameters

In [ ]:
sc.pp.neighbors(counts_wildtypes, n_neighbors=150, n_pcs=12, use_rep="X_scanorama")
sc.tl.umap(counts_wildtypes, min_dist=0.5)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (100)}):
    sc.pl.umap(counts_wildtypes, color=["batch", "phase", "Aqp4"])

In [ ]:
sc.pp.neighbors(counts_wildtypes, n_neighbors=50, n_pcs=12, use_rep="X_scanorama")
sc.tl.umap(counts_wildtypes, min_dist=0.5)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (100)}):
    sc.pl.umap(counts_wildtypes, color=["batch", "phase", "Aqp4"])

In [ ]:
sc.pp.neighbors(counts_wildtypes, n_neighbors=100, n_pcs=12, use_rep="X_scanorama")
sc.tl.umap(counts_wildtypes, min_dist=1)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (100)}):
    sc.pl.umap(counts_wildtypes, color=["batch", "phase", "Aqp4"])

In [ ]:
sc.pp.neighbors(counts_wildtypes, n_neighbors=100, n_pcs=8)
sc.tl.umap(counts_wildtypes, min_dist=0.5)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (100)}):
    sc.pl.umap(counts_wildtypes, color=["batch", "phase", "Aqp4"])

In [ ]:
sc.pp.neighbors(counts_wildtypes, n_neighbors=100, n_pcs=12, use_rep="X_scanorama")
sc.tl.umap(counts_wildtypes, min_dist=0.3)

In [ ]:
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (150)}):
    sc.pl.umap(counts_wildtypes, color=["batch", "phase", "Aqp4", "Cpe"], ncols=2)

### Re-calculate UMAP using the best parameters

In [ ]:
# re-compute UMAP with the best parameters
sc.pp.neighbors(counts_wildtypes, n_neighbors=100, n_pcs=12, use_rep="X_scanorama")  # n_pcs might be useless since I tell it to use X_scanorama
sc.tl.umap(counts_wildtypes, min_dist=0.5)
sc.pl.umap(counts_wildtypes, color=["batch", "phase"])

In [ ]:
counts_wildtypes.write_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected.h5ad')

In [ ]:
counts_wildtypes = sc.read_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected.h5ad')

## Clustering with DBSCAN. This allows us to then get only the cells belonging to the NCS lineage.

In [ ]:
clustering = DBSCAN(eps=0.5, min_samples=4).fit(counts_wildtypes.obsm["X_umap"])

In [ ]:
labels = clustering.labels_
no_clusters = len(np.unique(labels) )
no_noise = np.sum(np.array(labels) == -1, axis=0)
print('Estimated no. of clusters: %d' % no_clusters, 'Estimated no. of noise points: %d' % no_noise, sep='\n')

In [ ]:
labels_char = [str(label) for label in labels]
counts_wildtypes.obs['dbscan_clusters'] = labels_char
sc.pl.umap(counts_wildtypes, color="dbscan_clusters", use_raw=False)

In [ ]:
# remove the cell that does not cluster
counts_wildtypes = counts_wildtypes[counts_wildtypes.obs.dbscan_clusters!="-1"]

In [ ]:
fig, ax = plt.subplots()
sc.pl.umap(counts_wildtypes, color='batch', title="Batch correction for the 6 experiments", show=False, ax=ax)
fig.set_tight_layout(True)

## Check DBSCAN clusters identities

In [ ]:
marker_genes_svz_dict = dict({
    "qNSC1": ["Aqp4", "Cxcl14"],
    "qNSC2": ["Cpe", "S100a6"],
    "aNSC1": ["Btg2", "Egr1"],
    "aNSC2": ["Top2a", "Egfr", "Ascl1"],
    "TAP": ["Mki67", "Ube2c"],
    "NB": ["Dcx", "Stmn2"],
    "Neurons": ["Rbfox3", "Calb2", "Nefl", "Th"],
    "Microglia": ["Itgam", "Ptprc"],
    "Oligodendrocytes": ["Cldn11"],
    "OPC": ["Olig1", "Olig2", "Lhfpl3", "Cntn1"],
    "Ependymal cells": ["Cald1", "Sod3"],
    "Olfactory Ensheathing cells": ["Apod"],
    "pericyte": ["Foxd1", "Colec12"],
    "Endothelial cells": ["Prom1", "Cldn5", "Pecam1"],
})
marker_genes_svz_dict

In [ ]:
marker_genes_svz_nolineage_dict = dict({
    "Neurons": ["Rbfox3", "Calb2", "Nefl", "Th"],
    "Microglia": ["Itgam", "Ptprc"],
    "Oligodendrocytes": ["Cldn11"],
    "OPC": ["Olig1", "Olig2", "Lhfpl3", "Cntn1"],
    "Ependymal cells": ["Cald1", "Sod3"],
    "Olfactory Ensheathing cells": ["Apod"],
    "pericyte": ["Foxd1", "Colec12"],
    "Endothelial cells": ["Prom1", "Cldn5", "Pecam1"],
})
marker_genes_svz_nolineage_dict

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white', color_map="cividis_r")
sc.tl.dendrogram(counts_wildtypes, groupby="dbscan_clusters", optimal_ordering=True)
sc.pl.tracksplot(counts_wildtypes, marker_genes_svz_dict, groupby='dbscan_clusters', dendrogram=False)

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white', color_map="cividis_r")
sc.tl.dendrogram(counts_wildtypes[counts_wildtypes.obs.dbscan_clusters != "0"], groupby="dbscan_clusters", optimal_ordering=True)
sc.pl.tracksplot(counts_wildtypes[counts_wildtypes.obs.dbscan_clusters != "0"], marker_genes_svz_nolineage_dict, groupby='dbscan_clusters', dendrogram=False)

In [ ]:
sc.pl.umap(counts_wildtypes, color="dbscan_clusters", use_raw=False)

## Find the root to use for pseudotime

In [ ]:
proj_2d = px.scatter(counts_wildtypes.obsm["X_umap"], x=0, y=1, width=1000, height=1000, color=counts_wildtypes.obs.phase, labels={'color': 'Phase'},
                                                        hover_data={"Index": range(0, len(counts_wildtypes.obsm["X_umap"]))})
proj_2d.write_html(file=f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/DBSCAN_phase.html')

In [ ]:
proj_2d = px.scatter(counts_wildtypes.obsm["X_umap"], x=0, y=1, width=1000, height=1000, color=counts_wildtypes.to_df().loc[:, "Cxcl14"], labels={'color': 'Cxcl14'},
                                                        hover_data={"Index": range(0, len(counts_wildtypes.obsm["X_umap"]))})
proj_2d.write_html(file=f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/DBSCAN_Cxcl14.html')

In [ ]:
proj_2d = px.scatter(counts_wildtypes.obsm["X_umap"], x=0, y=1, width=1000, height=1000, color=counts_wildtypes.to_df().loc[:, "Aqp4"], labels={'color': 'Aqp4'},
                                                        hover_data={"Index": range(0, len(counts_wildtypes.obsm["X_umap"]))})
proj_2d.write_html(file=f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/DBSCAN_Aqp4.html')

In [ ]:
proj_2d = px.scatter(counts_wildtypes.obsm["X_pca"], x=0, y=1, width=1000, height=1000, color=counts_wildtypes.to_df().loc[:, "Aqp4"], labels={'color': 'Aqp4'},
                                                        hover_data={"Index": range(0, len(counts_wildtypes.obsm["X_umap"]))})
proj_2d.write_html(file=f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/DBSCAN_Aqp4_pca.html')

In [ ]:
# Pick the root cell for pseudotime via diffusion maps. Picked via first cell on PCA plot
root = counts_wildtypes.obsm["X_umap"][13092]

fig, ax = plt.subplots()
ax.plot(root[0], root[1], marker="o", color="red", markersize=5)
sc.pl.umap(counts_wildtypes, color="dbscan_clusters", use_raw=False, ax=ax, title="Clusters from DBSCAN and root cell")

fig.set_tight_layout(True)

In [ ]:
# Pick the root cell for pseudotime via diffusion maps. Picked via 'first' cell on UMAP
root = counts_wildtypes.obsm["X_umap"][4053]

fig, ax = plt.subplots()
ax.plot(root[0], root[1], marker="o", color="red", markersize=5)
sc.pl.umap(counts_wildtypes, color="dbscan_clusters", use_raw=False, ax=ax, title="Clusters from DBSCAN and root cell")

fig.set_tight_layout(True)

In [ ]:
# subset the data to contain only cells from the NCS lineage
counts_wildtypes_lineage = counts_wildtypes[counts_wildtypes.obs.dbscan_clusters == "0"]
sc.pl.umap(counts_wildtypes_lineage, color="dbscan_clusters", use_raw=False)

In [ ]:
sc.pp.scale(counts_wildtypes_lineage)
sc.pp.pca(counts_wildtypes_lineage, n_comps=12)

In [ ]:
sc.pl.pca(counts_wildtypes_lineage, color="phase")

In [ ]:
proj_2d = px.scatter(counts_wildtypes_lineage.obsm["X_pca"], x=0, y=1, width=1000, height=1000, color=counts_wildtypes_lineage.to_df().loc[:, "Aqp4"], labels={'color': 'Aqp4'},
                                                        hover_data={"Index": range(0, len(counts_wildtypes_lineage.obsm["X_umap"]))})
proj_2d.write_html(file=f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/DBSCAN_Aqp4_pca_lineage.html')

In [ ]:
# root picked via PCA of just the lineage
# get the index for the root, but now in the whole dataset
root_lineage_index = counts_wildtypes.obs.index.to_list().index(counts_wildtypes_lineage.obs.iloc[13317].name)
root_lineage = counts_wildtypes.obsm["X_umap"][root_lineage_index]

fig, ax = plt.subplots()
ax.plot(root_lineage[0], root_lineage[1], marker="o", color="red", markersize=5)
sc.pl.umap(counts_wildtypes, color="dbscan_clusters", use_raw=False, ax=ax, title="Clusters from DBSCAN and root cell")

fig.set_tight_layout(True)

In [ ]:
root_lineage_index

## Calculate diffusion pseudotime

In [ ]:
# Let's use the root found via PCA of the whole dataset and Aqp4 expression
# get the index for the root, but now in the lineage subset
counts_wildtypes_lineage.obs.index.to_list().index(counts_wildtypes.obs.iloc[13092].name)

In [ ]:
# root = counts_wildtypes_lineage.obsm["X_umap"][4639]
# counts_wildtypes_lineage.uns['iroot'] = root[0]
counts_wildtypes_lineage.uns['iroot'] = 11452

counts_wildtypes_lineage.X = counts_wildtypes_lineage.layers["lognorm"]

# re-calculate PCA and neighbours for the lineage only
sc.tl.pca(counts_wildtypes_lineage, svd_solver='arpack', n_comps=12)
sc.pp.neighbors(counts_wildtypes_lineage, n_neighbors=100, use_rep="X_scanorama", method="gauss")
# generate a diffusion map
sc.tl.diffmap(counts_wildtypes_lineage, n_comps=12)
# calculate diffusion pseudotime
sc.tl.dpt(counts_wildtypes_lineage, n_branchings=0, n_dcs=10)

In [ ]:
"""#This block calculates pseudotime based on the UMAP, but due to the large number of cycling cells I computed it based on the PCA
# generate a diffusion map
sc.tl.diffmap(counts_wildtypes_lineage, n_comps=15)
# calculate diffusion pseudotime
counts_wildtypes_lineage.uns['iroot'] = 11452
sc.tl.dpt(counts_wildtypes_lineage, n_dcs=15, n_branchings=0)"""

In [ ]:
with plt.rc_context({"figure.figsize": (5, 4), "figure.dpi": (150)}):
    sc.pl.umap(counts_wildtypes_lineage, color="dpt_pseudotime", use_raw=False, color_map="inferno")
    sc.pl.diffmap(counts_wildtypes_lineage, color="dpt_pseudotime", use_raw=False, color_map="inferno")
    sc.pl.pca(counts_wildtypes_lineage, color="dpt_pseudotime", color_map="inferno")

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sc.pl.umap(counts_wildtypes_lineage, color='dpt_pseudotime', use_raw=False, color_map="inferno",
    title="Diffusion pseudotime along the lineage", show=False, ax=ax)
fig.set_tight_layout(True)
# fig.savefig("../figures/Diffusion pseudotime along the lineage.png", dpi=400)

In [ ]:
sc.pl.umap(counts_wildtypes_lineage, color=["dpt_pseudotime", "phase", "Aqp4", "Dcx"], use_raw=True)

In [ ]:
counts_wildtypes.obs = counts_wildtypes.obs.join(counts_wildtypes_lineage.obs.dpt_pseudotime)
counts_wildtypes

In [ ]:
sc.pl.umap(counts_wildtypes, color=["dpt_pseudotime", "phase", "Aqp4", "Dcx"], use_raw=False)

In [ ]:
sc.pl.umap(counts_wildtypes, color=["dpt_pseudotime", "n_counts", "n_genes", "pct_counts_mt", "Aqp4", "Dcx"], use_raw=False)

In [ ]:
counts_wildtypes.write_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime.h5ad')

In [ ]:
counts_wildtypes = sc.read_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime.h5ad')

In [ ]:
# inspect the embedding with sleepwalk
umap_coords = counts_wildtypes_lineage.obsm["X_umap"]
nr, nc = umap_coords.shape
r_umap_matrix = robjects.r.matrix(umap_coords, nrow=nr, ncol=nc)

pca_coords = counts_wildtypes_lineage.obsm["X_pca"]
nr, nc = pca_coords.shape
r_feature_matrix = robjects.r.matrix(pca_coords, nrow=nr, ncol=nc)
r_sleepwalk.sleepwalk(r_umap_matrix, r_feature_matrix)

## Scrublet (run before; kept to show why it was run. Do not rerun this section: the underlying data has changed)

In [ ]:
adata_raw = counts_wildtypes.raw.to_adata()
adata_raw

In [ ]:
sce.pp.scrublet(adata=adata_raw)

In [ ]:
sce.pl.scrublet_score_distribution(adata=adata_raw)

In [ ]:
test_adata = counts_wildtypes
test_adata.obs = test_adata.obs.join(adata_raw.obs[["doublet_score", "predicted_doublet"]])
test_adata.obs["doublet"] = np.where(test_adata.obs.predicted_doublet, "yes", "no")

In [ ]:
sc.pl.umap(test_adata, color=["doublet_score", "doublet"])

## Check cell types based on the labs' previous found markers

In [ ]:
marker_genes = ["Aqp4", "S100b", "Slc1a3", "Egfr", "Ascl1", "Mki67", "Dlx1", "Dlx2", "Dcx", "Rbfox3", "Gria1", "Mag", "Mog", "Mbp", "Sox10"]
marker_celltypes = ("Astrocytes", "Astrocytes", "Astrocytes", "aNSCs", "aNSCs_TAPs", "aNSCs_TAPs", "Neuroblasts", "Neuroblasts", "Neuroblasts", 
                    "Neurons", "Neurons", "Oligodendrocytes", "Oligodendrocytes", "Oligodendrocytes", "Oligodendrocytes")

In [ ]:
pseudotime_expression = counts_wildtypes_lineage.raw.to_adata()[:, counts_wildtypes_lineage.raw.to_adata().var_names]
pseudotime_expression = pseudotime_expression.to_df()
pseudotime_expression = pseudotime_expression[pseudotime_expression.columns.intersection(marker_genes)]
pseudotime_expression["pTime"] = counts_wildtypes_lineage.obs.dpt_pseudotime
pseudotime_expression["phase"] = counts_wildtypes_lineage.obs.phase
pseudotime_expression = pseudotime_expression.sort_values("pTime").drop_duplicates(subset=["pTime"])

In [ ]:
pseudotime_expression

In [ ]:
# number of cells over pseudotime:
fig, ax = plt.subplots(1,1, figsize=(7, 3))
sns.kdeplot(data=pseudotime_expression, x="pTime", bw=0.05, ax=ax, c="lightslategray")
ax = sns.rugplot(data=pseudotime_expression, x="pTime", alpha=.05, c="k", hue="phase")
sns.move_legend(ax, "upper center", title="Cell cycle\n   phase", alignment="center")
ax.grid(False)
ax.set_ylabel("Cell density")
ax.set_xlim(0, 1)
fig.set_tight_layout(True)
# fig.savefig("../figures/Cell density along pseudotime.png", dpi=400)

In [ ]:
#pseudotime_bspline = scipy.interpolate.make_interp_spline(pseudotime_expression["pseudotime"], pseudotime_expression["Aqp4"], k=13)
#x_new = np.linspace(start=0, stop=1, num=300)
#y_new = pseudotime_bspline(x_new)

z = np.polyfit(pseudotime_expression["pTime"], pseudotime_expression["Aqp4"], 3)
p = np.poly1d(z)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(15, 5))
ax.plot(pseudotime_expression["pTime"], pseudotime_expression["Aqp4"])
ax.plot(pseudotime_expression["pTime"], p(pseudotime_expression["pTime"]))

In [ ]:
markers_df = pd.melt(pseudotime_expression, id_vars=["pTime"], value_vars=marker_genes)
markers_df

In [ ]:
# we generate a color palette with Seaborn.color_palette()
pal = sns.color_palette(palette='cividis', n_colors=16)

# in the sns.FacetGrid class, the 'hue' argument is the one that will be represented by colors with 'palette'
g = sns.FacetGrid(markers_df, row='variable', hue='variable', aspect=15, height=0.75, palette=pal)

# then we add the expression of each cell for each gene
g.map(plt.plot, 'pTime', 'value',
      alpha=1, linewidth=1.5)

# here we add a horizontal line for each plot
g.map(plt.axhline, y=0,
      lw=2, clip_on=False)

# we loop over the FacetGrid figure axes (g.axes.flat) and add the gene name and cell type as text with the right color
# notice how ax.lines[-1].get_color() enables you to access the last line's color in each matplotlib.Axes
for i, ax in enumerate(g.axes.flat):
      ax.set_xlim(-0.19, 1.19)
      ax.text(-0.15, 1.0, marker_genes[i],
            fontweight='bold', fontsize=15,
            color=ax.lines[-1].get_color())

for i, ax in enumerate(g.axes.flat):
    ax.text(1.02, 1.0, marker_celltypes[i],
            fontweight='bold', fontsize=15,
            color=ax.lines[-1].get_color())

# we use matplotlib.Figure.subplots_adjust() function to get the subplots to overlap (overlap not working, but =0 works)
g.fig.subplots_adjust(hspace=0)

# eventually we remove axes titles, yticks and spines
g.set_titles("")
g.set(yticks=[])
g.despine(bottom=True, left=True)

plt.setp(ax.get_xticklabels(), fontsize=12, color="#333333")
plt.xlabel('Pseudotime', fontweight='bold', fontsize=15, color="#333333")

plt.show()

In [ ]:
marker_celltypes

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white', color_map="cividis_r")
sc.tl.dendrogram(counts_wildtypes, groupby="dbscan_clusters", optimal_ordering=True)
sc.pl.tracksplot(counts_wildtypes, marker_genes, groupby='dbscan_clusters', dendrogram=False)